##Importación de Librerías y Carga de Datos

Asegúrate de tener instalada la librería `mlxtend` (`pip install mlxtend`). Necesitamos cruzar las ventas con el catálogo de productos para que las reglas tengan nombres legibles y no solo IDs numéricos.

In [1]:
import pandas as pd
import numpy as np

# Librerías para Reglas de Asociación
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
# Nota: Usamos fpgrowth en lugar de apriori porque es mucho más rápido y eficiente computacionalmente para 60,000 registros.

# Carga de datos
ventas = pd.read_csv('../data/processed/fact_ventas.csv', sep=';')
productos = pd.read_csv('../data/processed/dim_producto.csv', sep=';')

# Unimos para obtener el nombre de la categoría y del producto
df_ventas = ventas.merge(productos[['id_producto', 'nombre', 'categoria']], on='id_producto', how='inner')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


##Preparación de las Transacciones (Canastas)

Los algoritmos de asociación no entienden filas de ventas individuales; necesitan "canastas" (una lista de los productos que se llevaron en una sola transacción).

Nota analítica: Puedes hacer el análisis a nivel de `nombre` (producto específico) o a nivel de `categoria`. A nivel de producto suele generar miles de reglas débiles, mientras que a nivel de categoría genera reglas más estratégicas. Haremos el ejemplo a nivel de categoría (si prefieres producto, solo cambia la columna en el groupby).

In [2]:
# Agrupamos los elementos comprados en la misma boleta (id_venta)
# Usamos 'categoria' para obtener reglas de alto nivel (ej. "Snacks" -> "Bebidas")
transacciones = df_ventas.groupby('id_venta')['categoria'].apply(list).tolist()

# Verificamos cómo lucen las primeras 3 canastas
print("Muestra de transacciones:")
for i in range(3):
    print(transacciones[i])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Muestra de transacciones:
['Abarrotes', 'Abarrotes', 'Abarrotes', 'Abarrotes']
['Bebés']
['Frutas y Verduras', 'Abarrotes', 'Frutas y Verduras', 'Abarrotes']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

##Codificación de la Matriz (One-Hot Encoding)

Transformamos esa lista de listas en una matriz booleana gigante donde cada columna es una categoría y cada fila es una boleta (True si la llevó, False si no).

In [3]:
# Instanciamos el codificador
te = TransactionEncoder()
matriz_transacciones = te.fit_transform(transacciones)

# Convertimos a DataFrame para mlxtend
basket = pd.DataFrame(matriz_transacciones, columns=te.columns_)

print(f"Dimensiones de la matriz de canastas: {basket.shape}")

Dimensiones de la matriz de canastas: (34755, 12)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

##Generación de Itemsets Frecuentes y Reglas

Aquí entra la magia. Primero buscamos combinaciones frecuentes y luego derivamos las reglas calculando el Soporte, la Confianza y el Lift.

In [4]:
# 1. Encontrar itemsets frecuentes
# min_support = 0.01 significa que la combinación debe aparecer en al menos el 1% de todas las boletas
frecuentes = fpgrowth(basket, min_support=0.01, use_colnames=True)

# 2. Generar las reglas de asociación
# Filtramos directamente por aquellas que tengan un Lift mayor a 1.2 (indica una fuerte asociación positiva)
reglas = association_rules(frecuentes, metric="lift", min_threshold=1.2)

# Limpiamos y ordenamos para mejor legibilidad
reglas = reglas.sort_values("lift", ascending=False)

# Convertimos frozensets a strings planos para que se exporten bien a CSV y Power BI
reglas['antecedents'] = reglas['antecedents'].apply(lambda x: ', '.join(list(x)))
reglas['consequents'] = reglas['consequents'].apply(lambda x: ', '.join(list(x)))

# Redondeamos las métricas
columnas_metricas = ['support', 'confidence', 'lift', 'leverage', 'conviction']
reglas[columnas_metricas] = reglas[columnas_metricas].round(3)

print("Top 5 Reglas de Asociación (Por Lift):")
print(reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Top 5 Reglas de Asociación (Por Lift):
                     antecedents                   consequents  support  \
8                Snacks y Dulces              Bebidas, Lácteos    0.012   
5               Bebidas, Lácteos               Snacks y Dulces    0.012   
20                 Carnes y Aves  Abarrotes, Frutas y Verduras    0.014   
17  Abarrotes, Frutas y Verduras                 Carnes y Aves    0.014   
19             Frutas y Verduras      Carnes y Aves, Abarrotes    0.014   

    confidence   lift  
8        0.110  4.252  
5        0.454  4.252  
20       0.143  3.969  
17       0.383  3.969  
19       0.108  3.189  


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

##Exportación para Power BI

Para que tu equipo pueda visualizar esto en el dashboard de Power BI, exportamos el listado de reglas.

In [6]:
# Seleccionamos las columnas más relevantes para el reporte
export_pbi_asociacion = reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

ruta_salida_asociacion = '../data/processed/reglas_asociacion.csv'
export_pbi_asociacion.to_csv(ruta_salida_asociacion, sep=';', index=False)
print(f"Reglas de asociación exportadas exitosamente a {ruta_salida_asociacion}")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Reglas de asociación exportadas exitosamente a reglas_asociacion.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag